# 92 — Compare Geode, streamer, and nodal source waveforms at common shots

This notebook queries the SQLite catalog for common shot positions/times and compares nodal versus Geode/streamer empirical waveforms and spectra.

In [ ]:
from pathlib import Path
import sqlite3
import json
import math
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from obspy import read, Stream, Trace, UTCDateTime
from scipy.signal import detrend
from scipy.signal.windows import tukey

# Adjust if your repo lives somewhere else
REPO_ROOT = Path.cwd()
for parent in [Path.cwd(), *Path.cwd().parents]:
    if (parent / "lib").exists():
        REPO_ROOT = parent
        break

import sys
LIB = REPO_ROOT / "lib"
if str(LIB) not in sys.path:
    sys.path.append(str(LIB))

try:
    from segy_tools.gather import plot_wiggle_gather_from_stream, plot_wiggle_gather_from_segy
except Exception as e:
    print("Could not import segy_tools plotting helpers:", e)

In [ ]:
# ------------------------------------------------------------------
# Project paths
# ------------------------------------------------------------------
PROJECT_ROOT = Path("/Volumes/tachyon/LBSSP_DATA")
CATALOG_DB = PROJECT_ROOT / "catalog" / "lbssp_shot_catalog.sqlite"
OUTPUT_ROOT = PROJECT_ROOT / "source_waveform_analysis"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

EXPORT_CSV = True
EXPORT_FIGURES = True

# Short window: closest approximation to early body-wave pulse.
SHORT_PRE_S = 0.005
SHORT_POST_S = 0.030

# Effective window: source + near-source ground coupling, but not full 0.5 s ground roll if avoidable.
EFFECTIVE_PRE_S = 0.010
EFFECTIVE_POST_S = 0.150

# Diagnostic window: includes ground roll/airwave/site response for inspection.
DIAGNOSTIC_PRE_S = 0.020
DIAGNOSTIC_POST_S = 0.500

WINDOWS = {
    "short_body_pulse": (SHORT_PRE_S, SHORT_POST_S),
    "effective_wavelet": (EFFECTIVE_PRE_S, EFFECTIVE_POST_S),
    "diagnostic_long": (DIAGNOSTIC_PRE_S, DIAGNOSTIC_POST_S),
}

NFFT = 8192
FMAX_PLOT = 500.0
TAPER_ALPHA = 0.5

assert CATALOG_DB.exists(), f"Catalog database not found: {CATALOG_DB}"
print(CATALOG_DB)

In [ ]:
def connect_catalog(db_path=CATALOG_DB):
    conn = sqlite3.connect(db_path)
    conn.execute("PRAGMA busy_timeout = 30000")
    return conn


def read_table(conn, table):
    try:
        return pd.read_sql(f"SELECT * FROM {table}", conn)
    except Exception as e:
        print(f"Could not read {table}: {e}")
        return pd.DataFrame()


def parse_time(x):
    if pd.isna(x) or x in ("", None):
        return None
    try:
        return UTCDateTime(str(x))
    except Exception:
        try:
            return UTCDateTime(pd.to_datetime(x).to_pydatetime())
        except Exception:
            return None


def station_x_from_code(station):
    """Position-coded stations are cm along line: 08808 -> 88.08 m."""
    try:
        return float(str(station)) / 100.0
    except Exception:
        return np.nan


def load_stream_for_event(files_df, event_id, component=None, instrument_system=None):
    q = files_df[files_df["event_id"] == event_id].copy()
    if component is not None:
        q = q[q["component"].astype(str).str.upper() == component.upper()]
    if instrument_system is not None and "instrument_system" in q.columns:
        q = q[q["instrument_system"] == instrument_system]

    # Prefer 3C MiniSEED if no specific component requested.
    if component is None:
        q0 = q[(q["file_type"] == "mseed") | (q["component"].astype(str).str.upper() == "3C")]
        if len(q0):
            q = q0

    if len(q) == 0:
        raise FileNotFoundError(f"No files for event_id={event_id}, component={component}")

    # Read first available waveform file.
    row = q.iloc[0]
    f = Path(row["file_path"])
    if not f.exists():
        raise FileNotFoundError(f)
    return read(str(f))


def nearest_trace(st, source_x_m, component="Z", max_offset_m=None):
    """Return nearest Trace to source_x_m, optionally choosing a component suffix."""
    rows = []
    for i, tr in enumerate(st):
        cha = tr.stats.channel
        if component is not None and not cha.upper().endswith(component.upper()):
            continue
        x = station_x_from_code(tr.stats.station)
        if not np.isfinite(x):
            continue
        off = abs(x - source_x_m)
        if max_offset_m is not None and off > max_offset_m:
            continue
        rows.append((off, x, i, tr))
    if not rows:
        return None, None
    rows = sorted(rows, key=lambda r: r[0])
    return rows[0][3], {"offset_m": rows[0][0], "receiver_x_m": rows[0][1], "trace_index": rows[0][2]}


def get_pick_time_for_trace(picks_df, event_id, tr, preferred_phase="P"):
    if picks_df.empty:
        return None
    q = picks_df[picks_df["event_id"] == event_id].copy()
    for col, value in [("station", tr.stats.station), ("channel", tr.stats.channel)]:
        if col in q.columns:
            q = q[q[col].astype(str) == str(value)]
    if "phase" in q.columns and preferred_phase is not None:
        qp = q[q["phase"].astype(str).str.upper() == preferred_phase.upper()]
        if len(qp):
            q = qp
    for col in ["pick_time_utc", "pick_time", "consensus_time", "time"]:
        if col in q.columns and q[col].notna().any():
            return parse_time(q.iloc[0][col])
    return None


def extract_window_around_time(tr, center_time, pre_s, post_s, taper_alpha=TAPER_ALPHA):
    if center_time is None:
        raise ValueError("center_time is None")
    t1 = center_time - pre_s
    t2 = center_time + post_s
    pulse = tr.copy().trim(t1, t2, pad=True, fill_value=0)
    x = pulse.data.astype(float)
    if len(x) < 4:
        raise ValueError("window too short")
    x = detrend(x, type="linear")
    x = x - np.nanmean(x)
    w = tukey(len(x), alpha=taper_alpha)
    xt = x * w
    return pulse, x, xt


def amplitude_spectrum(x, dt, nfft=NFFT, normalize=True):
    freq = np.fft.rfftfreq(nfft, dt)
    amp = np.abs(np.fft.rfft(x, n=nfft))
    if normalize and np.nanmax(amp) > 0:
        amp = amp / np.nanmax(amp)
    return freq, amp


def plot_waveform_spectrum(x, xt, dt, title="", fmax=FMAX_PLOT):
    t = np.arange(len(x)) * dt
    freq, amp = amplitude_spectrum(xt, dt)
    fig, axes = plt.subplots(2, 1, figsize=(9, 6), constrained_layout=True)
    axes[0].plot(t, x, label="raw window")
    axes[0].plot(t, xt, label="tapered")
    axes[0].set_xlabel("Time since window start (s)")
    axes[0].set_ylabel("Amplitude")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    axes[0].set_title(title)
    axes[1].plot(freq, amp)
    axes[1].set_xlim(0, fmax)
    axes[1].set_xlabel("Frequency (Hz)")
    axes[1].set_ylabel("Normalized amplitude")
    axes[1].grid(True, alpha=0.3)
    return fig, freq, amp


def event_label(row):
    parts = [str(row.get("line", "")), str(row.get("source_type", "")), str(row.get("source_x_m", "")), str(row.get("event_id", ""))]
    return " | ".join([p for p in parts if p and p != "nan"])

In [ ]:
conn = connect_catalog()
shot_events = read_table(conn, "shot_events")
shot_files = read_table(conn, "shot_gather_files")
picks = read_table(conn, "picks")

shot_events["source_x_m"] = pd.to_numeric(shot_events.get("source_x_m", np.nan), errors="coerce")
shot_events["instrument_system_norm"] = shot_events.get("instrument_system", "").astype(str).str.lower()
print(shot_events.shape, shot_files.shape, picks.shape)

## Find common shot positions

In [ ]:
position_tol_m = 0.25

positioned = shot_events[shot_events["source_x_m"].notna()].copy()
positioned["source_x_round"] = (positioned["source_x_m"] / position_tol_m).round() * position_tol_m

common = (
    positioned.groupby([c for c in ["line", "source_x_round", "source_type"] if c in positioned.columns], dropna=False)
    .agg(
        n_events=("event_id", "count"),
        n_systems=("instrument_system_norm", "nunique"),
        systems=("instrument_system_norm", lambda x: ",".join(sorted(set(map(str, x))))),
    )
    .reset_index()
    .sort_values(["n_systems", "n_events"], ascending=False)
)

display(common.head(100))
common.to_csv(OUTPUT_ROOT / "common_shot_positions_by_system.csv", index=False)

## Select one common shot group

In [ ]:
# Pick the first group with at least two instrument systems, or edit manually.
choice = common[common["n_systems"] >= 2].head(1)
if len(choice) == 0:
    choice = common.head(1)
assert len(choice), "No positioned events found"
choice = choice.iloc[0]
print(choice)

sel = positioned.copy()
if "line" in positioned.columns and "line" in choice.index:
    sel = sel[sel["line"].astype(str) == str(choice["line"])]
sel = sel[np.abs(sel["source_x_m"] - float(choice["source_x_round"])) <= position_tol_m]
if "source_type" in sel.columns and "source_type" in choice.index and pd.notna(choice["source_type"]):
    sel = sel[sel["source_type"].astype(str) == str(choice["source_type"])]

display(sel[[c for c in ["event_id", "instrument_system", "line", "source_x_m", "source_type", "shot_time_utc", "metadata_match_status"] if c in sel.columns]])

## Extract nearest traces and compare spectra

In [ ]:
records = []
figdir = OUTPUT_ROOT / "cross_system_comparisons"
figdir.mkdir(parents=True, exist_ok=True)

fig, axes = plt.subplots(2, 1, figsize=(10, 7), constrained_layout=True)

for _, ev in sel.iterrows():
    event_id = str(ev["event_id"])
    system = str(ev.get("instrument_system", "unknown"))
    source_x_m = float(ev["source_x_m"])
    try:
        st = load_stream_for_event(shot_files, event_id, component=None)
    except Exception as e:
        print("skip", event_id, e)
        continue
    tr, ninfo = nearest_trace(st, source_x_m, component="Z")
    if tr is None:
        continue
    center = get_pick_time_for_trace(picks, event_id, tr)
    if center is None:
        center = tr.stats.starttime
    try:
        pulse, x, xt = extract_window_around_time(tr, center, EFFECTIVE_PRE_S, EFFECTIVE_POST_S)
    except Exception as e:
        print("window skip", event_id, e)
        continue
    dt = pulse.stats.delta
    t = np.arange(len(xt)) * dt - EFFECTIVE_PRE_S
    freq, amp = amplitude_spectrum(xt, dt)
    label = f"{system} {event_id} off={ninfo['offset_m']:.1f}m"
    axes[0].plot(t, xt / (np.nanmax(np.abs(xt)) or 1), label=label, alpha=0.8)
    axes[1].plot(freq, amp, label=label, alpha=0.8)
    records.append({
        "event_id": event_id,
        "instrument_system": system,
        "source_x_m": source_x_m,
        "receiver_x_m": ninfo["receiver_x_m"],
        "offset_m": ninfo["offset_m"],
        "channel": tr.stats.channel,
    })

axes[0].set_xlabel("Time relative to pick/start (s)")
axes[0].set_ylabel("Normalized amplitude")
axes[0].grid(True, alpha=0.3)
axes[0].legend(fontsize=8)
axes[1].set_xlim(0, FMAX_PLOT)
axes[1].set_xlabel("Frequency (Hz)")
axes[1].set_ylabel("Normalized amplitude spectrum")
axes[1].grid(True, alpha=0.3)
axes[1].legend(fontsize=8)
fig.suptitle("Common shot comparison")
fig.savefig(figdir / "common_shot_comparison_example.png", dpi=150)
plt.show()

comparison_df = pd.DataFrame(records)
display(comparison_df)
comparison_df.to_csv(OUTPUT_ROOT / "common_shot_comparison_example.csv", index=False)